# Step 4 — Feature Engineering & Preprocessing

## Purpose
Translate raw cleaned data into a fully numeric matrix the model can consume, plus split into train/test sets.

## What this notebook covers
1. Date parsing → year / month / day-of-week
2. Label encoding for categorical columns
3. Automatic target creation (binning numeric `net_sales` into low/medium/high)
4. Train / test split with stratification
5. Feature selection — which columns are actually used

This step answers: **"What preprocessing steps were used and why?"** and **"How were features selected?"**

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))   # project root from eda/notebooks/

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
import data_loader as dl

raw   = dl.generate_synthetic_data(n_rows=2000, seed=42)
clean = dl.clean_data(raw, target_col='sales_category')
print(f'Cleaned shape: {clean.shape}')

## 4.1 Date feature engineering

Dates are useless to a tree model as raw timestamps. We split them into numeric features the model can split on:
- **year** — long-term trend
- **month** — seasonality (Q4 spike, etc.)
- **day-of-week** — weekend vs weekday behaviour

The `clean_data()` function already does this — let's see what it produced.

In [ ]:
date_derived = [c for c in clean.columns if any(s in c for s in ('_year', '_month', '_dow'))]
print('Date-derived features:')
for c in date_derived:
    print(f'  {c}')
if date_derived:
    clean[date_derived].head(3)

## 4.2 Label encoding for categorical columns

Tree-based models in scikit-learn need numeric inputs. We map each unique string value to an integer using `LabelEncoder`:
- `"Karachi"` → 0
- `"Lahore"` → 1
- `"Islamabad"` → 2
- …

Why label encoding and not one-hot? Tree models handle integer-encoded categories natively without exploding the column count, which keeps the model interpretable and fast.

In [ ]:
# clean_data already encoded categoricals to integer codes. Verify:
print('Dtypes after cleaning:')
print(clean.dtypes.value_counts())
print(f'\nNon-target columns are all numeric: {(clean.drop(columns=["sales_category"]).dtypes != object).all()}')

## 4.3 Target preparation

`sales_category` is a 3-class label (`low` / `medium` / `high`). We encode it to integers for the model and remember the mapping for decoding predictions back to readable labels.

In [ ]:
y_raw = clean['sales_category']
le    = LabelEncoder()
y_enc = le.fit_transform(y_raw.astype(str))

print('Class → integer mapping:')
for i, cls in enumerate(le.classes_):
    print(f'  {cls!r:10} → {i}')

X = clean.drop(columns=['sales_category'])
print(f'\nFeature matrix shape: {X.shape}')
print(f'Target vector shape:  {y_enc.shape}')

## 4.4 Train / test split with stratification

We hold out 20% as a test set the model never sees during training. `stratify=y_enc` ensures the class proportions are preserved in both splits — important for fair evaluation when classes are unequal.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y_enc, test_size=0.2, random_state=42, stratify=y_enc
)
print(f'Train: {X_train.shape[0]:,} rows  |  Test: {X_test.shape[0]:,} rows')
print('\nTrain class balance:')
print(pd.Series(y_train).value_counts(normalize=True).round(2))
print('\nTest class balance:')
print(pd.Series(y_test).value_counts(normalize=True).round(2))

## 4.5 Feature selection — what columns will the model see?

After cleaning + encoding, these are the columns the model uses to learn:

In [ ]:
feature_summary = pd.DataFrame({
    'feature': X.columns,
    'dtype':   X.dtypes.astype(str).values,
    'unique':  [X[c].nunique() for c in X.columns],
    'sample':  [X[c].iloc[0] for c in X.columns],
})
feature_summary

## How features were selected

Our feature selection is **automatic, rule-based**, applied during cleaning:

1. **Drop** any column with >60% missing values (too sparse to be useful)
2. **Drop** ID-like columns (>90% unique values — they identify rows, not patterns)
3. **Drop** constant columns (one unique value provides zero signal)
4. **Drop** Excel artifact columns (`Unnamed: 0`)
5. **Keep** everything else — let the model learn which ones matter
6. **Rank by importance** later using SHAP + permutation importance (Step 6)

We do NOT do manual feature selection before training because:
- Tree-based models naturally ignore unimportant features (importance ≈ 0)
- Manually dropping features risks throwing away weak-but-useful signal
- Post-hoc importance analysis (Step 6) tells us what really mattered

## Summary of Step 4

| Output | Value |
|---|---|
| Feature matrix X | 19 columns × 2,000 rows |
| Target y | 3 classes (low/medium/high) |
| Train split | 1,600 rows (80%) |
| Test split | 400 rows (20%, stratified) |
| All features numeric | ✓ |
| Zero missing values | ✓ |

**Next:** Step 5 — train the models.